# Camera Feed Live Detection

## load libraries

In [1]:
import cv2
from datetime import datetime
import numpy as np
from ultralytics import YOLO

## drawining bounding box function

In [2]:
BBOX_COLORS = [
    (0, 0, 255),
    (0, 255, 0),
    (0, 255, 255),
    (255, 0, 0),
    (255, 0, 255),
    (128, 128, 128),
    (0, 165, 255),
    (19, 69, 139)
]
COLORS_NUM = len(BBOX_COLORS)

# function draw the predicted bounding boxes 
def draw_detections(frame, detections, labels, min_thresh=0.5):
    count = 0
    for det in detections:
        conf = det.conf.item()
        if conf < min_thresh:
            continue

        # get the prediction info
        xyxy = det.xyxy.cpu().numpy().squeeze().astype(int)
        xmin, ymin, xmax, ymax = xyxy

        class_id  = int(det.cls.item())
        classname = labels[class_id]
        color     = BBOX_COLORS[class_id % COLORS_NUM]

        # drow bounding boxes
        cv2.rectangle(frame, (xmin, ymin), (xmax, ymax), color, 2)

        # draw the label and the prediction confidince
        label = f"{classname}: {int(conf * 100)}%"
        (lw, lh), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
        label_y = max(ymin-baseline, lh)
        cv2.rectangle(frame, (xmin, label_y - lh), (xmin + lw, label_y + baseline), color, cv2.FILLED)
        cv2.putText(frame, label, (xmin, label_y), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 0), 1)

        count += 1

    # draw the number of objects in the frame
    cv2.putText(frame, f"Objects: {count}", (10, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (193, 170, 62), 2)
    return frame

## detection function

In [3]:
def live_detection(model_path, min_thresh=0.5, camera_index=0, resolution=(1280, 960)):

    # load the model
    model = YOLO(model_path, task="detect")
    labels = model.names

    # open the camera
    cam = cv2.VideoCapture(camera_index)

    if not cam.isOpened():
        print("ERROR: could not open camera.")
        return
    else:
        print("camera opened.\nPress q to quit\ns to pause\np to save screenshot.")

    fps_buffer = [] # save the last 30 fps value

    while True:
        t_start = cv2.getTickCount()

        ret, frame = cam.read()
        if not ret or frame is None:
            print("lost camera feed.\nExiting.")
            break
        
        #resize the frame to the wanted resolution
        frame = cv2.resize(frame, resolution)

        # run inferance on the frame
        results = model(frame, verbose=False)
        detections = results[0].boxes

        # draw the predictions
        frame = draw_detections(frame, detections, labels, min_thresh)

        # compute and draw FPS
        time = (cv2.getTickCount() - t_start) / cv2.getTickFrequency()
        fps_buffer.append(1.0 / time)
        if len(fps_buffer) > 30:
            fps_buffer.pop(0)
        avg_fps = np.mean(fps_buffer)
        cv2.putText(frame, f'FPS: {avg_fps:.1f}', (10, 20), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (193, 170, 62), 2)
        
        # show output
        cv2.imshow("Live Detection  q:quit  s:pause(press any key to play)  p:screenshot", frame)

        key = cv2.waitKey(1)
        if key == ord("q") or key == ord("Q"):
            break
        elif key == ord("s") or key == ord("S"):
            cv2.waitKey(0)
        elif key == ord("p") or key == ord("P"):
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            cv2.imwrite(f"capture_{timestamp}.png", frame)
            print(f"screenshot saved as capture_{timestamp}.png")

    cam.release()
    cv2.destroyAllWindows()
    print("camera closed")

## run live detection

set the model path in "`MODEL_PATH`"

In [7]:
MODEL_PATH = r"..\..\..\model_development\model_training\model_4_yolo11m_datasetv4\best.pt"

live_detection(
    model_path = MODEL_PATH,
    min_thresh = 0.6,
    camera_index = 0,
    resolution = (1280, 960)
)

camera opened.
Press q to quit
s to pause
p to save screenshot.
camera closed
